# RQ3 — Diagnostic des NaN de la baseline (pli 0, graine 2026)

`adaptation/diag_nan.py` : même réseau, mêmes données, même pli et même graine que la baseline, 1 000 itérations
au plus par variante, journal par itération. A = baseline (fp16, Dice par patch, sans clipping ni warmup),
B = fp32, C = Dice sur le lot, D = unscale_ + clip_grad_norm_(12) + warmup sur 500 itérations.
Au premier poids ou moment non fini : lot, état avant le pas et gradients du pas dans `diag_nan/<variante>/rupture/`.

In [ ]:
# Environnement : GPU obligatoire (accélérateur « GPU T4 »), sinon arrêt immédiat avec un message clair.
import hashlib, json, os, re, shutil, subprocess, sys, time, zipfile
from pathlib import Path
import torch
print("torch", torch.__version__, "| CUDA", torch.version.cuda, "| GPU",
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "aucun")
if not torch.cuda.is_available() and os.environ.get("RQ3_GPU", "1") == "1":
    raise RuntimeError("AUCUN GPU DISPONIBLE : dans le panneau de droite, Settings > Accelerator > « GPU T4 x2 », "
                       "puis relancer (Save Version > Save & Run All). Rien n'a été lancé.")
INPUT = Path(os.environ.get("RQ3_INPUT", "/kaggle/input"))      # datasets attachés
WORKING = Path(os.environ.get("RQ3_WORKING", "/kaggle/working"))  # toutes les sorties
SCRATCH = Path(os.environ.get("RQ3_TMP", "/tmp/rq3"))            # liens vers les données, jamais dans les sorties
EXPECTED_COMMIT = None  # commit attendu dans VERSION.json du dataset de code (None : pas de contrôle)

In [ ]:
# Code du projet : dataset privé projet-avc-code, cherché récursivement sous /kaggle/input.
def code_version(candidate):
    if candidate.suffix == ".zip":
        with zipfile.ZipFile(candidate) as archive:
            return json.loads(archive.read("VERSION.json")) if "VERSION.json" in archive.namelist() else None
    return json.loads(candidate.read_text())

candidates = sorted(INPUT.rglob("projet_avc_code.zip")) + sorted(
    path for path in INPUT.rglob("VERSION.json") if (path.parent / "adaptation" / "train.py").is_file())
found = [(path, code_version(path)) for path in candidates]
found = [(path, version) for path, version in found if version]
if not found:
    raise FileNotFoundError("Dataset projet-avc-code introuvable sous /kaggle/input : Add Input > Datasets > "
                            "« projet-avc-code » (onglet Shared with you), puis relancer.")
matching = [(path, version) for path, version in found if EXPECTED_COMMIT in (None, version["commit"])]
if not matching:
    raise RuntimeError(f"VERSION.json : commit {sorted({v['commit'] for _, v in found})} au lieu de {EXPECTED_COMMIT}. "
                       "Mettre l'entrée projet-avc-code à sa dernière version (menu ⋮ de l'entrée), puis relancer.")
source, VERSION = matching[0]
print("Code :", source, VERSION)
REPO = WORKING / "PROJET_AVC"
if not REPO.exists():
    if source.suffix == ".zip":
        zipfile.ZipFile(source).extractall(REPO)
    else:  # Kaggle a décompressé l'archive : copie de l'arborescence
        shutil.copytree(source.parent, REPO)
if os.environ.get("RQ3_PIP", "1") == "1":
    pip = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "monai==1.6.1", "torchio==1.2.1",
                          "nibabel==5.4.2", "pyarrow"])
    if pip.returncode:
        raise RuntimeError("Installation de MONAI / TorchIO impossible : activer Internet "
                           "(Settings > Internet on, compte vérifié par téléphone), puis relancer.")

In [ ]:
# Données C1 : dataset privé atlas-c1-processed, cherché récursivement ; liens vers les dossiers de sujets.
tables = sorted(INPUT.rglob("subjects.parquet"))
if len(tables) != 1:
    raise FileNotFoundError(f"subjects.parquet trouvé {len(tables)} fois sous /kaggle/input (1 attendu) : "
                            "ajouter le dataset atlas-c1-processed (Add Input > Datasets > Shared with you).")
TABLE = tables[0]
DATA = SCRATCH / "processed"
DATA.mkdir(parents=True, exist_ok=True)
subjects = sorted(INPUT.rglob("qc.json"))
if not subjects:  # archives restées compressées : extraction hors des sorties
    for archive in sorted(INPUT.rglob("processed_partie*.zip")):
        zipfile.ZipFile(archive).extractall(SCRATCH / "c1_extrait")
    subjects = sorted((SCRATCH / "c1_extrait").rglob("qc.json"))
for qc in subjects:
    link = DATA / qc.parent.name
    if not link.exists():
        link.symlink_to(qc.parent, target_is_directory=True)
if not (DATA / "subjects.parquet").exists():
    (DATA / "subjects.parquet").symlink_to(TABLE)
print(len(subjects), "sujets prétraités ; table", TABLE)
if not subjects:
    raise FileNotFoundError("Aucun dossier de sujet (qc.json) dans atlas-c1-processed")
# Même découpage que le code : splits.json du dataset (à côté de subjects.parquet) identique à celui du dépôt.
data_splits = next((folder / "splits.json" for folder in [TABLE.parent, *TABLE.parents]
                    if folder != INPUT and INPUT in folder.parents and (folder / "splits.json").is_file()), None)
if data_splits is None:
    raise FileNotFoundError("splits.json absent du dataset atlas-c1-processed")
digest = lambda path: hashlib.sha256(Path(path).read_bytes()).hexdigest()
assert digest(data_splits) == digest(REPO / "splits.json"), "splits.json du dataset différent de celui du code"

In [ ]:
# Diagnostic : les quatre variantes l'une après l'autre ; sortie complète aussi dans diag_nan/diag_nan.log.
!mkdir -p /kaggle/working/diag_nan && cd {REPO} && python -u -m adaptation.diag_nan --data-root {DATA} --output /kaggle/working/diag_nan --num-workers 4 2>&1 | tee /kaggle/working/diag_nan/diag_nan.log

In [ ]:
# Fichiers à récupérer (onglet Output).
print((Path("/kaggle/working/diag_nan") / "resume.json").read_text())
!find /kaggle/working/diag_nan -type f -exec ls -la {} +